# Notebook 3: Period Finding

## Why do we search for periods?

Many astronomical objects vary in brightness on a regular cycle: **eclipsing binaries** dim when one star passes in front of the other, **pulsating stars** expand and contract rhythmically, and **rotating stars** show spots that come in and out of view. The *period* of this variability is one of the most fundamental measurements we can make — it tells us about the orbital dynamics, physical size, and evolutionary state of the system.

But finding the period from real data is tricky! The observations are unevenly spaced (we can only observe at night, and weather intervenes), noisy, and may span years. We need specialized algorithms to tease out the periodic signal.

## Common types of periodic variable stars

| Type | Typical Period | What's happening |
|------|---------------|------------------|
| Eclipsing binary | hours to days | Two stars orbit each other; we see dips when one blocks the other |
| Pulsating white dwarf | 2–20 minutes | The WD's surface oscillates, changing its brightness |
| AM CVn | 5–65 minutes | Ultra-compact binary with helium transfer between two white dwarfs |
| Delta Scuti | 0.5–6 hours | A-type star pulsating in multiple modes |
| RR Lyrae | 0.2–1 day | Old giant star pulsating — great distance indicators |

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

import ztf_tools

## Step 1: Load a light curve

We'll use the longest-period ultracompact binary from `interesting_objects.csv` — a good test case because its ~56-minute period is well-resolved by ZTF's cadence.

Alternatively, you can grab your own candidate from the catalog — see the commented-out code below.

In [ ]:
import pandas as pd

objects = pd.read_csv('interesting_objects.csv', comment='#')

# === Option A: Pick the longest-period object from interesting_objects.csv ===
source = objects.iloc[-1]
ra, dec = source['ra'], source['dec']

# === Option B: Grab your own candidate from the catalog ===
# (Uncomment the lines below and comment out Option A)
# catalog = ztf_tools.load_catalog('/ztf/catalogs/box_least_squares', max_files=200)
# candidates = ztf_tools.filter_candidates(
#     catalog, min_significance=50, max_period=180/1440
# )
# candidates = candidates[candidates['bls_power'] < 100]  # avoid the brightest/weirdest
# source = candidates.sample(1).iloc[0]  # pick a random candidate
# ra, dec = source['ra'], source['dec']

# Look up the catalog period from BLS
catalog_info = ztf_tools.lookup_period(ra=ra, dec=dec)
catalog_period = catalog_info['period']

print(f'Source: {source.get("name", f"RA={ra:.4f}, Dec={dec:.4f}")}')
print(f'Catalog period: {catalog_period:.6f} days ({catalog_period*1440:.2f} min)')
print(f'BLS power: {catalog_info["significance"]:.1f}')

In [ ]:
lc = ztf_tools.get_lightcurve(ra=ra, dec=dec)

# Use whichever filter has the most data
best_filt = max(lc.keys(), key=lambda k: len(lc[k]['time']))
t = lc[best_filt]['time']
y = lc[best_filt]['flux']
dy = lc[best_filt]['flux_err']

print(f'Using {best_filt}-band: {len(t)} data points')

## Step 2: Compute a Lomb-Scargle periodogram

The **Lomb-Scargle periodogram** computes the "power" of a sinusoidal signal at each trial frequency. High power means the data matches a sinusoid at that frequency well. We'll use the implementation in `astropy.timeseries.LombScargle`.

We search a range of frequencies corresponding to periods from ~1 minute up to 1 day:

In [ ]:
# Define the frequency grid
min_freq = 1.0     # 1/day → period = 1 day
max_freq = 1440.0  # 1440/day → period = 1 minute

ls = LombScargle(t, y, dy)
frequency, power = ls.autopower(
    minimum_frequency=min_freq,
    maximum_frequency=max_freq
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(frequency, power, lw=0.5, color='steelblue')
ax.set_xlabel('Frequency (1/day)')
ax.set_ylabel('Lomb-Scargle Power')
ax.set_title('Periodogram')

# Mark the best frequency
best_freq = frequency[np.argmax(power)]
ax.axvline(best_freq, color='red', ls='--', alpha=0.7, label=f'Best: {best_freq:.2f} /day')
ax.legend()
fig.tight_layout()
plt.show()

best_period = 1.0 / best_freq
print(f'Best period from periodogram: {best_period:.6f} days ({best_period*1440:.2f} min)')

## Step 3: Compare with the catalog period

The catalog already ran a (much more thorough) Lomb-Scargle search. Let's see how our quick result compares:

In [ ]:
print(f'Catalog period:      {catalog_period:.6f} days ({catalog_period*1440:.2f} min)')
print(f'Our best period:     {best_period:.6f} days ({best_period*1440:.2f} min)')
print(f'Ratio:               {best_period / catalog_period:.4f}')

# Check if they're the same (or a harmonic)
ratio = best_period / catalog_period
nearest_harmonic = round(ratio)
if abs(ratio - nearest_harmonic) < 0.05:
    if nearest_harmonic == 1:
        print('→ Great match! Same period.')
    else:
        print(f'→ Found a {nearest_harmonic}× harmonic of the catalog period.')
else:
    print('→ Different period — try zooming into the periodogram near the catalog frequency.')

## Step 4: Phase-fold at different periods

The best way to tell if a period is correct is to **phase-fold** the data at that period. If the period is right, you'll see a clean, repeating pattern. If it's wrong, the points will look scattered.

Let's compare folding at the right period vs. a slightly wrong one:

In [ ]:
wrong_period = catalog_period * 1.07  # 7% off

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Correct period
ph_right = ztf_tools.phase_fold(t, catalog_period)
bp, bf, be = ztf_tools.weighted_bin(ph_right, y, dy, n_bins=50)
for offset in range(3):
    ax1.errorbar(bp + offset, bf, be, ls='-', marker='.', color='steelblue', alpha=0.8)
ax1.set_title(f'Correct period: {catalog_period:.6f} d')
ax1.set_xlabel('Phase')
ax1.set_ylabel('Flux')
ax1.set_xlim(0, 3)

# Wrong period
ph_wrong = ztf_tools.phase_fold(t, wrong_period)
bp2, bf2, be2 = ztf_tools.weighted_bin(ph_wrong, y, dy, n_bins=50)
for offset in range(3):
    ax2.errorbar(bp2 + offset, bf2, be2, ls='-', marker='.', color='coral', alpha=0.8)
ax2.set_title(f'Wrong period: {wrong_period:.6f} d (7% off)')
ax2.set_xlabel('Phase')
ax2.set_ylabel('Flux')
ax2.set_xlim(0, 3)

fig.suptitle('Right period vs. wrong period', fontsize=13)
fig.tight_layout()
plt.show()

See the difference? With the correct period, the data folds into a clean pattern. With the wrong period, the structure is washed out and the binned points look noisy.

## A note on BLS (Box Least Squares)

The Lomb-Scargle algorithm fits sinusoids, which works great for smoothly-varying objects (pulsators, ellipsoidal variables). But for **eclipsing systems**, where the brightness drops sharply during an eclipse and is roughly constant otherwise, a different algorithm called **Box Least Squares (BLS)** does better.

BLS searches for periodic box-shaped dips in the light curve. The pre-computed BLS results are available in `/ztf/catalogs/box_least_squares`, and you can look up the BLS period for any source with:

```python
info = ztf_tools.lookup_period(ra=ra, dec=dec)  # defaults to BLS catalog
print(f'BLS period: {info["period"]:.6f} days, power: {info["significance"]:.1f}')
```

You can also compare with the LS catalog:

```python
info_ls = ztf_tools.lookup_period(ra=ra, dec=dec, catalog_dir='/ztf/catalogs/lomb_scargle')
```

If you're looking for eclipsing binaries or transiting planets, BLS is the way to go!

## Summary

In this notebook you learned:

1. **Why period finding matters** for understanding variable stars
2. How to compute a **Lomb-Scargle periodogram** with astropy
3. How to **compare** your results with the pre-computed catalog
4. How to **visually validate** a period by phase-folding
5. When to use **BLS** instead of Lomb-Scargle

## Next steps

Here are some ideas for further exploration:

- **Cross-match with Gaia**: Use `astropy.coordinates.SkyCoord.match_to_catalog_sky()` to find Gaia parallaxes and proper motions for your candidates. This can tell you if they're nearby white dwarfs or distant giant stars.
- **Color information**: Compare the g-band and r-band light curves. The ratio of variability amplitudes in different filters gives clues about the temperature of the varying component.
- **Multi-mode pulsators**: Some stars pulsate at multiple periods simultaneously. Try subtracting the best-fit sinusoid and running the periodogram again on the residuals.
- **Explore BLS candidates**: Load the BLS catalog and look for eclipsing systems with sharp, well-defined eclipses.